### Tools

Models can request to call tools that perform tasks such as data fetching from a database, searching the web, or running code. 

Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions. (often a JSON schema)
2. A function or co-routine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]
model = init_chat_model("groq:llama-3.3-70b-versatile")
model

In [ ]:
from langchain.tools import tool

@tool
def get_weather(place: str) -> str:
    '''Get the weather of specified place'''
    return f"It's sunny in {place}"

model_with_tool = model.bind_tools([get_weather])

In [ ]:
response = model_with_tool.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    if 'type' in tool_call:
        print(f"Type: {tool_call['type']}")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
messages: list[HumanMessage | AIMessage] = [HumanMessage(content="What's the weather like in Boston?")]
response = model_with_tool.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_message = model_with_tool.invoke(messages)

print(final_message.text)